<a href="https://colab.research.google.com/github/Username772/carisurg-portfolio/blob/main/notebooks/Week_5/Week5_Tutorial2_Data_Profiling_STUDENT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div style="border-radius:10px;overflow:hidden;font-family:Aptos,Calibri,Segoe UI,sans-serif;border:1px solid #e6e9ee">
  <div style="background:#0A2540;padding:18px 22px;color:#fff">
    <div style="font-size:12px;letter-spacing:2px;color:#F4B942;font-weight:700">CARISURG · MEDTECH PATHWAYS · WEEK 5 · TUTORIAL 2</div>
    <div style="font-size:24px;font-weight:700;margin-top:4px">Data Profiling</div>
    <div style="font-size:14px;color:#cdd6df;margin-top:6px">Measuring missingness, types, distributions and outliers across 225 features.</div></div>
  <div style="background:#F4B942;color:#0A2540;padding:6px 22px;font-size:12px;font-weight:700;letter-spacing:1px">STUDENT NOTEBOOK</div></div>

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Goal
Turn eyeballing into measurement. With 225 columns we profile the **structured** features in detail and the **chief-complaint** family in aggregate. Output: a data-quality issues table for the Tutorial 5 memo.

## 1 · Setup

In [ ]:
# Run this cell first. These are the three libraries we use all week.
import numpy as np                 # numerical helpers (NaN, medians, etc.)
import pandas as pd                # tables / DataFrames — our main tool
import matplotlib.pyplot as plt    # plotting

# Let pandas show more of a wide table when we print it:
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 140)

print("Environment ready · pandas", pd.__version__)

In [ ]:
# ── Schema map ────────────────────────────────────────────────────────────────
# This dataset has ~225 columns, so we never list them by hand. We sort them into
# "families" once, then refer to the families by name for the rest of the week.

TARGET = "esi"   # Emergency Severity Index: 1 (most urgent) .. 5 (least). Our triage label.

# Vital-sign columns measured at the front door:
VITALS = ["triage_vital_hr", "triage_vital_sbp", "triage_vital_dbp", "triage_vital_rr",
          "triage_vital_o2", "triage_vital_temp", "triage_glucose"]
# Who the patient is (some of these are fairness-sensitive — handle with care):
DEMOGRAPHICS = ["age", "gender", "ethnicity", "race", "lang", "religion",
                "maritalstatus", "employstatus", "insurance_status"]
# Administrative / arrival details:
ADMIN = ["dep_name", "arrivalmode", "arrivalmonth", "arrivalday", "arrivalhour_bin"]
# OUTCOMES of the visit — known only AFTER triage, so they must never be model inputs:
LEAKAGE = ["disposition", "previousdispo"]

def classify_columns(df):
    """Sort the DataFrame's columns into families and return them in a dictionary."""

    # Helper: from a wish-list of names, keep only the ones that really exist in df.
    # (A plain function instead of a lambda, so it is easy to read.)
    def keep_present(wanted):
        present = []
        for col in wanted:
            if col in df.columns:
                present.append(col)
        return present

    # The ~200 chief-complaint columns all start with "cc_", so we find them by prefix:
    chief_complaints = []
    for col in df.columns:
        if col.startswith("cc_"):
            chief_complaints.append(col)

    families = {
        "target":           keep_present([TARGET]),
        "vitals":           keep_present(VITALS),
        "demographics":     keep_present(DEMOGRAPHICS),
        "admin":            keep_present(ADMIN),
        "leakage":          keep_present(LEAKAGE),
        "chief_complaints": chief_complaints,
    }
    return families

In [ ]:
# Reference ranges for general adult triage. Each entry is (low, high, unit).
# NOTE: temperature is in FAHRENHEIT in this dataset (≈98.6 normal), not Celsius!
NORMAL_RANGES = {"triage_vital_hr": (60,100,"bpm"), "triage_vital_sbp": (90,140,"mmHg"),
    "triage_vital_dbp": (60,90,"mmHg"), "triage_vital_rr": (12,20,"/min"),
    "triage_vital_o2": (95,100,"%"), "triage_vital_temp": (97.0,99.5,"F"), "triage_glucose": (70,140,"mg/dL")}

# "Plausible" bounds are much wider than normal — anything OUTSIDE these is treated as a
# data error (e.g. a heart rate of 5). Each entry is (low, high).
PLAUSIBLE = {"age": (0,120), "esi": (1,5), "triage_vital_hr": (20,250), "triage_vital_sbp": (50,300),
    "triage_vital_dbp": (20,200), "triage_vital_rr": (4,60), "triage_vital_o2": (50,100),
    "triage_vital_temp": (86,110), "triage_glucose": (20,800)}

In [ ]:
from pathlib import Path
# Google Colab: mount Drive then point DATA_PATH at the file (it is ~55 MB, give it a few seconds).
#   from google.colab import drive; drive.mount("/content/drive")
#   DATA_PATH = "/content/drive/MyDrive/CariSurg/yaleemmlc_admissionprediction_triage.csv"
DATA_PATH = Path("/content/drive/MyDrive/ColabNotebooks/CariSurg_Healthcare_AI/Week5-AI-Assisted_Triage:Data_Exploration(Part1_of_2)/yaleemmlc_admissionprediction_triage.csv")
if not Path(DATA_PATH).exists():
    raise FileNotFoundError(f"Could not find {DATA_PATH}. Set DATA_PATH (Colab) or place the CSV beside this notebook.")
df = pd.read_csv(DATA_PATH, index_col=0)   # index_col=0 drops the unnamed export index column
print(f"Loaded {df.shape[0]:,} encounters x {df.shape[1]} columns")

In [ ]:
fam = classify_columns(df)
# "structured" = the non-chief-complaint columns (everything not starting with "cc_").
structured = [col for col in df.columns if not col.startswith("cc_")]

## 2 · Missingness (structured columns)

Chief-complaint flags are 0/1, not missing — so we profile missingness on the structured columns.

<div style="border-left:4px solid #F4B942;background:#f7f9fb;padding:10px 14px;border-radius:4px;font-family:Aptos,Calibri,sans-serif"><b style="color:#F4B942">✏️ YOUR TASK</b><br>Measure the <b>fraction missing</b> in each structured column, turn it into a percentage, and show only the columns that actually have gaps (worst first). The cell gives you the steps and the function names — you write the code.</div>

In [ ]:
# .isna() marks every cell True/False for "is this missing?".
# GOAL: a percentage-missing per structured column, worst first, gaps only.

# TODO 1 — FRACTION MISSING PER COLUMN
# HINT: df[structured].isna() gives True/False; taking .mean() of True/False = the fraction missing.
missing_fraction = df[structured].isna().mean()      # <- replace: add .mean() to get one fraction per column

# TODO 2 — TURN INTO A PERCENTAGE, worst first
# HINT: multiply by 100 and .round(1), then .sort_values(ascending=False).
missing_percent = (missing_fraction * 100).round(1)
missing_percent = missing_percent.sort_values(ascending=False)

# TODO 3 — SHOW ONLY COLUMNS WITH GAPS
# HINT: filter a Series with a mask -> missing_percent[missing_percent > 0]
missing_percent[missing_percent > 0]

<div style="border-left:4px solid #6C5CE7;background:#f7f9fb;padding:10px 14px;border-radius:4px;font-family:Aptos,Calibri,sans-serif"><b style="color:#6C5CE7">🧩 PANDAS IN PLAIN ENGLISH</b><br><code>.isna()</code> turns the table into True/False — True wherever a cell is empty. Taking <code>.mean()</code> of True/False gives the <b>fraction</b> that are True, i.e. the share missing (True = 1, False = 0). And <code>missing_percent[missing_percent &gt; 0]</code> filters a Series with a True/False mask: it keeps only the rows where the condition holds.</div>

In [ ]:
# Missingness map of the structured block (a 226-col map is unreadable)
try:
    import missingno as msno              # Colab: !pip install missingno -q
    msno.matrix(df[structured]); plt.show()
except Exception:
    fig, ax = plt.subplots(figsize=(11,4))
    ax.imshow(df[structured].isna().values, aspect="auto", cmap="gray_r")
    ax.set_xticks(range(len(structured))); ax.set_xticklabels(structured, rotation=90, fontsize=7)
    ax.set_title("Missing cells — structured columns (dark = missing)"); plt.tight_layout(); plt.show()

<div style="border-left:4px solid #1B9AAA;background:#f7f9fb;padding:10px 14px;border-radius:4px;font-family:Aptos,Calibri,sans-serif"><b style="color:#1B9AAA">🩺 CLINICAL CONTEXT</b><br>Watch <code>esi</code> missingness especially: a row with no triage label cannot teach a triage model, so those rows are candidates for removal (Tutorial 3) — not imputation. You never invent a triage decision.</div>

## 3 · Dtype audit + target sanity

In [ ]:
print(df[structured].dtypes)
print("\nESI value counts (raw):"); print(df[TARGET].value_counts(dropna=False).sort_index())

<div style="border-left:4px solid #6C5CE7;background:#f7f9fb;padding:10px 14px;border-radius:4px;font-family:Aptos,Calibri,sans-serif"><b style="color:#6C5CE7">🧩 PANDAS IN PLAIN ENGLISH</b><br><code>.dtypes</code> reports what <i>kind</i> of value each column holds — <code>float64</code> for numbers, <code>object</code> (or <code>str</code>) when text has sneaked into a numeric column. <code>.value_counts()</code> tallies how often each distinct value appears; <code>dropna=False</code> adds a row for the NaNs so missing labels can’t hide; <code>.sort_index()</code> orders the tally by ESI level (1→5).</div>

## 4 · Distributions

In [ ]:
# Draw a histogram for each vital. (The instructor version also adds an ESI bar.)
vitals = fam["vitals"]
fig, axes = plt.subplots(2, 4, figsize=(15, 6))
panels = axes.ravel()           # flatten the 2x4 grid into a list of 8 panels

for panel, col in zip(panels, vitals):
    # TODO: turn df[col] into numbers with pd.to_numeric(..., errors="coerce"), drop NaNs,
    #       then call panel.hist(values, bins=30) and panel.set_title(col).
    values = pd.to_numeric(df[col], errors="coerce").dropna()
    panel.hist(values, bins=30)
    panel.set_title(col, fontsize=9)
    pass

esi_counts = df[TARGET].value_counts().sort_index()
panels[-1].bar(esi_counts.index, esi_counts.values)
panels[-1].set_title("ESI class balance")

plt.tight_layout(); plt.show()

<div style="border-left:4px solid #6C5CE7;background:#f7f9fb;padding:10px 14px;border-radius:4px;font-family:Aptos,Calibri,sans-serif"><b style="color:#6C5CE7">🧩 PANDAS IN PLAIN ENGLISH</b><br><code>pd.to_numeric(df[col], errors='coerce')</code> forces a column to numbers and quietly turns anything it can’t parse (like <code>'120bpm'</code>) into NaN instead of crashing — <code>coerce</code> = ‘don’t raise an error, just mark it missing’. <code>.dropna()</code> then drops those gaps so the histogram only plots real numbers. <code>zip(panels, vitals)</code> pairs panel 1 with vital 1, panel 2 with vital 2, and so on, so one loop fills the whole grid.</div>

In [ ]:
# Demographics — the fairness-sensitive view (raw categories)
for c in ["race","ethnicity","insurance_status"]:
    print(c, "->", df[c].value_counts(dropna=False).to_dict())

## 5 · Outliers — statistical vs clinically impossible

<div style="border-left:4px solid #F4B942;background:#f7f9fb;padding:10px 14px;border-radius:4px;font-family:Aptos,Calibri,sans-serif"><b style="color:#F4B942">✏️ YOUR TASK</b><br>Finish <code>outlier_report</code> so it returns two counts per vital: <b>statistical</b> outliers (the 1.5×IQR fence) and <b>clinically impossible</b> values (outside plausible bounds). The function names you need are in the hints.</div>

In [ ]:
def outlier_report(df, col):
    """Count two kinds of outlier in one numeric column."""
    x = pd.to_numeric(df[col], errors="coerce").dropna()   # numbers only

    # TODO 1 — STATISTICAL OUTLIERS (the 1.5 x IQR "fence")
    # HINT: q1 = x.quantile(0.25); q3 = x.quantile(0.75); iqr = q3 - q1
    # HINT: count them with ((x < q1 - 1.5*iqr) | (x > q3 + 1.5*iqr)).sum()  -- keep the parentheses
    q1 = x.quantile(0.25)
    q3 = x.quantile(0.75)
    iqr = q3 - q1
    low_fence = q1 - 1.5*iqr
    high_fence = q3 + 1.5*iqr
    iqr_outliers = ((x < low_fence) | (x > high_fence)).sum()

    # TODO 2 — CLINICALLY IMPOSSIBLE (outside the wide plausible bounds)
    # HINT: hard_low, hard_high = PLAUSIBLE.get(col, (-np.inf, np.inf))
    # HINT: count them with ((x < hard_low) | (x > hard_high)).sum()
    hard_low, hard_high = PLAUSIBLE.get(col, (-np.inf, np.inf))
    impossible = ((x < hard_low) | (x > hard_high)).sum()

    return {"iqr_outliers": iqr_outliers, "impossible": impossible}

report_rows = {}
for col in fam["vitals"]:
    report_rows[col] = outlier_report(df, col)
pd.DataFrame(report_rows).T

<div style="border-left:4px solid #6C5CE7;background:#f7f9fb;padding:10px 14px;border-radius:4px;font-family:Aptos,Calibri,sans-serif"><b style="color:#6C5CE7">🧩 PANDAS IN PLAIN ENGLISH</b><br>Two ideas here. <code>x.quantile(0.25)</code>/<code>0.75</code> are the 25th/75th percentiles; the IQR ‘fence’ flags statistically unusual values. Combining masks needs <b>parentheses</b> and <code>|</code> (‘or’): <code>(x &lt; low) | (x &gt; high)</code> is True wherever either holds. Finally <code>pd.DataFrame(report_rows).T</code> builds a table, and <code>.T</code> flips rows and columns so each vital becomes a row.</div>

## 6 · Relationship with the real target

No provisional label this time — `esi` is real. Which vitals and complaints move with acuity?

<div style="border-left:4px solid #F4B942;background:#f7f9fb;padding:10px 14px;border-radius:4px;font-family:Aptos,Calibri,sans-serif"><b style="color:#F4B942">✏️ YOUR TASK</b><br>Measure how each vital and each chief complaint <b>correlates with <code>esi</code></b>, and print the strongest. Remember the sign: a <b>negative</b> correlation means the value rises as <code>esi</code> falls — i.e. it tracks <i>higher</i> acuity.</div>

In [ ]:
# Make a numeric copy of the vitals so we can correlate them with ESI.
vitals_numeric = df[fam["vitals"]].copy()
for col in fam["vitals"]:
    vitals_numeric[col] = pd.to_numeric(vitals_numeric[col], errors="coerce")

# TODO 1 — VITALS vs ESI
# HINT: vitals_numeric.corrwith(df[TARGET]) gives one correlation per vital; .sort_values() orders them.
# HINT: print(vital_corr.round(3)) to read them (negative = tracks higher acuity).
vital_corr = vitals_numeric.corrwith(df[TARGET]).sort_values()
print("Vital vs ESI:")
print(vital_corr.round(3))

# TODO 2 — CHIEF COMPLAINTS vs ESI (they are already 0/1 numbers)
# HINT: df[fam["chief_complaints"]].corrwith(df[TARGET]).dropna().sort_values()
# HINT: .head(8) shows the 8 complaints most associated with LOW esi (high acuity).
cc_corr = df[fam["chief_complaints"]].corrwith(df[TARGET]).dropna().sort_values()
print("\n Chief Complaints vs ESI:")
print(cc_corr.head(8).round(3))


<div style="border-left:4px solid #6C5CE7;background:#f7f9fb;padding:10px 14px;border-radius:4px;font-family:Aptos,Calibri,sans-serif"><b style="color:#6C5CE7">🧩 PANDAS IN PLAIN ENGLISH</b><br><code>.copy()</code> makes a separate copy so editing it never touches the original <code>df</code>. <code>.corrwith(df[TARGET])</code> correlates <i>every</i> column against one target Series in a single step, returning one number per column between −1 and +1. A <b>negative</b> value means ‘as this vital goes up, <code>esi</code> goes down’ — and lower <code>esi</code> means more urgent.</div>

<div style="border-left:4px solid #C0392B;background:#f7f9fb;padding:10px 14px;border-radius:4px;font-family:Aptos,Calibri,sans-serif"><b style="color:#C0392B">⚠️ DATA / SAFETY NOTE</b><br>Correlations here are <b>directional sanity checks</b> on a real but US-sourced label, not model performance and not causal. Many <code>cc_</code> flags are near-zero-variance, so their correlations are unstable — note that, do not over-read them.</div>

## 7 · Data-quality issues table

In [ ]:
# TODO: one row per measured problem (missing esi, missing vitals, impossible values,
# Fahrenheit, sparse cc_, leakage, representativeness). Columns: issue, columns, action.
issues = [
    {"issue": "Missing ESI label", "columns": "esi", "action": "drop rows w/o target"},
    {"issue": "Missing ESI label", "columns": "esi", "action": "drop rows w/o target"},
    {"issue": "Missing ESI label", "columns": "esi", "action": "drop rows w/o target"}

]
pd.DataFrame(issues)

## 8 · Exercises
1. Complete `outlier_report` and the target-correlation cell.
2. Is vitals missingness related to acuity? Compare ESI for rows missing `triage_vital_o2` vs not.
3. How many `cc_` columns are effectively constant (<0.5% prevalence)? What does that imply for modelling?
4. Finish the issues table — every problem gets a row and a planned action.

In [ ]:
#Exercise 1
#Compare ESI for paients with and without oxygen saturation recorded

missing_o2 = df[df["triage_vital_o2"].isna()]
not_missing_o2 = df[df["triage_vital_o2"].notna()]

print("Average ESI when O2 is missing:")
print(missing_o2["esi"].mean())

print("\nAverage ESI when O2 is recorded:")
print(not_missing_o2["esi"].mean())


print("\nESI Summary Statistics:")
print(df.groupby(df["triage_vital_o2"].isna())["esi"].describe())

print("\nESI Distribution:")
print(df.groupby(df["triage_vital_o2"].isna())["esi"].value_counts().sort_index())



Observation:
There are no missing values in triage_vital_o2, so missingness could not be compared.
For patients with recorded oxygen saturation, the average ESI is 2.88, and most patients fall between ESI 2 and 3. This suggests that oxygen saturation is routinely measured regardless of acuity level.

In [ ]:
#Exercise 2
#How many cc_ columns are effectively constant (<0.5% prevalence)?

# Calculate prevalence of each chief complaint

cc_prevalence = df[fam["chief_complaints"]].mean()

constant_cc = cc_prevalence[cc_prevalence < 0.005]

print("Number of chief complaint columns with <0.5% prevalence:")
print(len(constant_cc))

print("\nExamples:")
print(constant_cc.head(10))

In [ ]:
#Exercise 3
# Calculate prevalence of each chief complaint (cc_) column
cc_prevalence = df[fam["chief_complaints"]].mean()

# Select those that appear in less than 0.5% of patients
rare_cc = cc_prevalence[cc_prevalence < 0.005]

print("Number of cc_ columns with <0.5% prevalence:")
print(len(rare_cc))

print("\nExamples:")
print(rare_cc.head(10))

**Chief Complaint Feature Sparsity (<0.5% prevalence)**                           
Many chief complaint variables occur in fewer than 0.5% of patients. These features are extremely sparse and are unlikely to contribute strong predictive power individually. They may introduce noise and should be considered for grouping or removal during modelling.

In [ ]:
#Exercise 4
#Finish the issues table — every problem gets a row and a planned action.

issues = [
    {
        "issue": "Missing ESI labels",
        "columns": "esi",
        "action": "Drop rows with missing target values"
    },
    {
        "issue": "Missing vital signs",
        "columns": "triage_vital_*",
        "action": "Assess missingness and impute or flag where appropriate"
    },
    {
        "issue": "Clinically impossible values",
        "columns": "Vitals",
        "action": "Remove or correct invalid observations"
    },
    {
        "issue": "Temperature stored in Fahrenheit",
        "columns": "triage_vital_temp",
        "action": "Document units and ensure consistent interpretation"
    },
    {
        "issue": "Sparse chief complaint features",
        "columns": "cc_*",
        "action": "Consider grouping or removing very low-prevalence features"
    },
    {
        "issue": "Potential data leakage",
        "columns": "disposition",
        "action": "Exclude from predictor variables"
    },
    {
        "issue": "US-specific dataset",
        "columns": "Insurance, demographics",
        "action": "Assess suitability before applying to Caribbean settings"
    }
]

pd.DataFrame(issues)

## 9 · Wrap-up
You have measured the quality. **Next — Tutorial 3: Cleaning**, where each issue gets a documented fix.